# First Letters screener - Colab T4 (zero setup)

Runs `gemini_lane/screen_scroll_rtx4060.py` on a free Colab T4 (16 GB): m7 snap, on-sheet score, Studio fp16 and A-hn fp16 (both depth orders).

**Usage:** `Runtime > Change runtime type > T4 GPU`, then `Run all` up to the *Setup* + *Helpers* cells, then run ONE scenario cell at a time (i -> ix).

**Privacy rules (hard):**
- Outputs land in `/content/scrollprize_private` (outside the git clone) and are NEVER committed or pushed.
- Only `montage.png`, `hotspots.png` and `summary.json` are zipped or backed up to Drive.
- Zero persistent CT: the screener streams Zarr chunks into RAM and discards them after every mesh.
- Do not commit this notebook with outputs: clear all outputs before saving to GitHub.

## 1. Setup: clone + lightweight deps

In [ ]:
import subprocess
import sys

BRANCH = "gemini/coverage-gpu-lane"
REPO_URL = "https://github.com/JorisViaudQuantAI/Scrollprize.git"
REPO_DIR = "/content/Scrollprize"

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: switch runtime to T4")
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "numcodecs", "safetensors", "tifffile", "opencv-python-headless", "requests", "zarr",
    ],
    check=True,
)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. Helpers (no side effects: definitions only)

In [ ]:
import json
import subprocess
import sys
import time
import zipfile
from pathlib import Path

try:
    from google.colab import drive as colab_drive
    from google.colab import files as colab_files
except ImportError:
    colab_drive = None
    colab_files = None

REPO_DIR = Path("/content/Scrollprize")
OUT_DIR = Path("/content/scrollprize_private")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_BACKUP_DIR = DRIVE_MOUNT / "MyDrive" / "Scrollprize_private"
ZIP_PATH = Path("/content/scrollprize_private_compact.zip")
ZIP_ALLOWLIST = frozenset({"montage.png", "hotspots.png", "summary.json"})

S3 = "https://vesuvius-challenge-open-data.s3.us-east-1.amazonaws.com"
SCROLLS = {
    "PHerc0306B": {
        "ct": f"{S3}/PHerc0306B/volumes/20250521133212-8.640um-1.2m-116keV-masked.zarr",
        "m7": f"{S3}/PHerc0306B/representations/predictions/surfaces/"
        "20250521133212-surface-20260413222639-surface-m7-L0-th0.2.zarr",
        "voxel_um": 8.64,
    },
    "PHerc0125": {
        "ct": f"{S3}/PHerc0125/volumes/20250821151825-9.362um-1.2m-113keV-masked.zarr",
        "m7": f"{S3}/PHerc0125/representations/predictions/surfaces/"
        "20250821151825-surface-20260413222639-surface-m7-L0-th0.2.zarr",
        "voxel_um": 9.362,
    },
    "PHerc1203": {
        "ct": f"{S3}/PHerc1203/volumes/20250820131727-9.362um-1.2m-113keV-masked.zarr",
        "m7": f"{S3}/PHerc1203/representations/predictions/surfaces/"
        "20250820131727-surface-20260413222639-surface-m7-L0-th0.2.zarr",
        "voxel_um": 9.362,
    },
    "PHerc0191": {
        "ct": f"{S3}/PHerc0191/volumes/20250821151635-9.362um-1.2m-113keV-masked.zarr",
        "m7": f"{S3}/PHerc0191/representations/predictions/surfaces/"
        "20250821151635-surface-20260413222639-surface-m7-L0-th0.2.zarr",
        "voxel_um": 9.362,
    },
    "PHerc0826": {
        "ct": f"{S3}/PHerc0826/volumes/20250821151701-9.362um-1.2m-113keV-masked.zarr",
        "m7": f"{S3}/PHerc0826/representations/predictions/surfaces/"
        "20250821151701-surface-20260413222639-surface-m7-L0-th0.2.zarr",
        "voxel_um": 9.362,
    },
}

GATE_MESH = (
    "https://raw.githubusercontent.com/SirProdigle/pherc0306b-first-letters/"
    "feabff6c54f95bd8e52a90e0a7a76881ee67d4cc/submission/mesh/"
    "PHerc0306B_20250521133212_w074own_touchup_V5Rc"
)
GATE_UV_CROP = "80:140,120:345"
PSCAMILLO_COMMIT = "66000516c97e043b696ce874f145e29638109c91"
PSCAMILLO_BASE = (
    f"https://raw.githubusercontent.com/pscamillo/vesuvius-eligible-meshes/{PSCAMILLO_COMMIT}/meshes/PHerc0125"
)
# Rank-1 Tier-1 untouched windings; Codex's z14144_w045 is deliberately excluded.
PHERC0125_TIER1 = ("z10544_w035", "z14144_w035", "z8736_w035", "z6944_w030", "z15936_w030")
PHERC0125_EXCLUDED = ("z14144_w045",)
# PHerc0125 Tier-1B CPU-triaged high-coherence windings (23.06 cm2, f_parallel 0.894-0.985).
PHERC0125_TIER1B = (
    "z9936_w035",
    "z9344_w015",
    "z11136_w035",
    "z15936_w035",
    "z8144_w030",
    "z14144_w050",
)
# Rank-2 clean S3 segments/raw sheets (46.66 cm2, 0% overlap with Codex A/B/C and Claude N6).
PHERC1203_CLEAN = (
    "auto_grown_20251005231446965",
    "auto_grown_20250930104534929",
    "auto_grown_20251005230118636",
    "auto_grown_20250929222256117",
    "auto_grown_20250930000321811",
    "auto_grown_20251005231446963",
)
# 7th clean PHerc1203 Sep-23 S3 sheet (3.79 cm2, f_parallel 0.929, 0% overlap with Codex/Claude/top-9).
PHERC1203_SEP23_CLEAN = ("auto_grown_20250923163217042",)
# PHerc1203 Tier-2 disjoint UV strips (6.82 cm2, f_parallel 0.786-0.966; one --uv-crop per invocation).
PHERC1203_TIER2_STRIPS = (
    ("auto_grown_20251005230830030", "0:165,0:68"),
    ("auto_grown_20251005221856743", "0:253,158:226"),
    ("auto_grown_20251005221856743", "0:253,0:62"),
)
PHERC1203_BASE = f"{S3}/PHerc1203/segments/raw"
# PHerc0191 & PHerc0826 spiral fits (CC BY-NC 4.0, Carson Rodrigues; unvalidated geometry), pinned HF revision.
HF_SPIRAL_REVISION = "63c641ba1dc794073208c24f7f88fa59c4f77621"
PHERC0191_BASE = (
    "https://huggingface.co/datasets/rodriguescarson/eligible-scroll-spiral-fits/resolve/"
    f"{HF_SPIRAL_REVISION}/2026-09-15_PHerc0191_slice-11600-12400_0-patch_w90/meshes/fitted"
)
PHERC0191_WINDINGS = ("w020", "w030", "w040", "w050")
PHERC0826_BASE = (
    "https://huggingface.co/datasets/rodriguescarson/eligible-scroll-spiral-fits/resolve/"
    f"{HF_SPIRAL_REVISION}/2026-09-15_PHerc0826_slice-8928-9728_0-patch/meshes/fitted"
)
# High-coherence PHerc0826 windings (7.76 cm2, f_parallel 0.820-0.926); skips low-coherence w020 and w027/w051.
PHERC0826_WINDINGS = ("w030", "w050", "w040")
PHERC0826_EXCLUDED = ("w020", "w027", "w051")
GATE_DETECTOR_KEYS = ("studio_desc", "studio_asc", "ahn_s0_asc", "ahn_s0_desc")


def build_screen_cmd(scroll, meshes, *, snap_passes, uv_crop=None, detectors="studio,ahn", num_slices=31):
    """Return the screener argv for a T4 (16 GB): Studio fp16 stride 48 Hann + A-hn fp16."""
    cfg = SCROLLS[scroll]
    cmd = [
        sys.executable, "gemini_lane/screen_scroll_rtx4060.py",
        "--scroll", scroll,
    ]
    for mesh in meshes:
        cmd += ["--mesh", mesh]
    cmd += [
        "--ct-zarr", cfg["ct"],
        "--m7-zarr", cfg["m7"],
        "--snap-passes", str(snap_passes),
        "--num-slices", str(num_slices),
        "--voxel-um", str(cfg["voxel_um"]),
        "--detectors", detectors,
        "--studio-stride", "48",
        "--studio-batch", "4",
        "--ahn-batch", "2",
        "--vram-fraction", "0.85",
        "--batch-pause-ms", "0",
        "--max-gpu-temp-c", "85",
        "--cache-mb", "1024",
        "--out-private-dir", str(OUT_DIR),
    ]
    if uv_crop:
        cmd += ["--uv-crop", uv_crop]
    return cmd


def run_streaming(cmd):
    """Run ``cmd`` inside the clone and stream its output to the cell."""
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    proc = subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"command failed (rc={code})")


def gate_summaries():
    return sorted(OUT_DIR.glob("PHerc0306B_*_r80-140_c120-345/summary.json"))


def show_gate():
    """Print the positive-control detector stats so you can decide whether the gate passed."""
    found = gate_summaries()
    if not found:
        raise FileNotFoundError("Run cell (i) first: no PHerc0306B gate summary found")
    detectors = json.loads(found[-1].read_text(encoding="utf-8"))["detectors"]
    for key in GATE_DETECTOR_KEYS:
        stats = detectors.get(key)
        print(key, "MISSING" if stats is None else {k: stats[k] for k in ("p99", "p995", "frac_gt_05", "valid_cm2")})
    return detectors


def require_gate(reviewed):
    """Refuse unread-scroll runs until the clean positive control ran and was reviewed by you."""
    detectors = show_gate()
    missing = [k for k in GATE_DETECTOR_KEYS if k not in detectors]
    if missing:
        raise RuntimeError(f"Gate incomplete, missing detectors: {missing}")
    if not reviewed:
        raise RuntimeError("Inspect the gate montage/stats, then set GATE_REVIEWED = True")


def package_private_outputs(out_dir=OUT_DIR, zip_path=ZIP_PATH):
    """Zip ONLY montage.png / hotspots.png / summary.json (never CT, maps or anything else)."""
    out_dir, zip_path = Path(out_dir), Path(zip_path)
    members = sorted(p for p in out_dir.rglob("*") if p.is_file() and p.name in ZIP_ALLOWLIST)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in members:
            zf.write(path, path.relative_to(out_dir).as_posix())
    return zip_path, [p.relative_to(out_dir).as_posix() for p in members]


def backup_to_drive(zip_path=ZIP_PATH):
    """Copy the compact zip to Google Drive (outside the git clone)."""
    if colab_drive is None:
        raise RuntimeError("Drive backup only works inside Colab")
    colab_drive.mount(str(DRIVE_MOUNT))
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    dest = DRIVE_BACKUP_DIR / Path(zip_path).name
    dest.write_bytes(Path(zip_path).read_bytes())
    return dest


def assert_repo_clean():
    """Fail if anything (e.g. private outputs) leaked into the git clone."""
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], capture_output=True, text=True, check=True
    ).stdout
    if status.strip():
        raise RuntimeError(f"Git clone is dirty, do NOT commit:\n{status}")
    print("git clone clean")


PREP_HECATE_SCRIPT = Path("gemini_lane") / "progress_prize_fp16" / "prepare_hecate_parity_crops.py"
BURST_SCRIPT = Path("gemini_lane") / "progress_prize_fp16" / "verify_upstream_gpu_burst.py"
HECATE_CROPS_DIR = OUT_DIR / "hecate_parity_crops"
HECATE_CROP_2P4 = "hecate_2p4um_crop.npy"
HECATE_CROP_9P6 = "hecate_9p6um_crop.npy"


def build_prepare_hecate_crops_cmd(crops_dir=HECATE_CROPS_DIR):
    """Stream the two tiny (<15 MB total) public uint8 Hecate parity crops outside git."""
    return [
        sys.executable, str(PREP_HECATE_SCRIPT),
        "--out-dir", str(crops_dir),
    ]


def build_burst_cmd(output, crops_dir=HECATE_CROPS_DIR, hecate_patch="main"):
    """Full Progress Prize fp16 parity burst: Studio + Hecate (2.4 um & 9.6 um, fwd/rev, 2D & 3D)."""
    crops_dir = Path(crops_dir)
    return [
        sys.executable, str(BURST_SCRIPT),
        "--output", str(output),
        "--stages", "studio,hecate",
        "--hecate-patch", hecate_patch,
        "--hecate-input-2p4", str(crops_dir / HECATE_CROP_2P4),
        "--hecate-input-9p6", str(crops_dir / HECATE_CROP_9P6),
        "--vram-fraction", "0.35",
        "--batch", "1",
        "--max-gpu-temp-c", "80",
        "--cooldown-margin-c", "5",
        "--cooldown-timeout-s", "600",
        "--studio-stride", "48",
    ]


def run_burst(hecate_patch="main", crops_dir=HECATE_CROPS_DIR):
    """1-click Progress Prize fp16 parity burst: prepare Hecate crops, then run studio+hecate."""
    for script in (PREP_HECATE_SCRIPT, BURST_SCRIPT):
        if not (REPO_DIR / script).is_file():
            print(f"{script.name} not found on this branch: pull a branch that ships it, then rerun")
            return None
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    crops_dir = Path(crops_dir)
    run_streaming(build_prepare_hecate_crops_cmd(crops_dir))
    output = OUT_DIR / time.strftime("upstream_gpu_burst_%Y%m%d_%H%M%S.json")
    run_streaming(build_burst_cmd(output, crops_dir=crops_dir, hecate_patch=hecate_patch))
    print(output.read_text(encoding="utf-8"))
    return output

## (i) Clean positive-control gate: `PHerc0306B`
Published touched-up mesh, `--uv-crop 80:140,120:345`, Studio fp16 + A-hn fp16, no snap. Inspect the montage, and only then set `GATE_REVIEWED = True` below.

In [ ]:
run_streaming(
    build_screen_cmd("PHerc0306B", [GATE_MESH], snap_passes=0, uv_crop=GATE_UV_CROP)
)
show_gate()

In [ ]:
GATE_REVIEWED = False  # set True ONLY after inspecting the 0306B positive-control montage/stats

## (ii) Rank-1 `PHerc0125` Tier-1 untouched windings
`z10544_w035`, `z14144_w035`, `z8736_w035`, `z6944_w030`, `z15936_w030` (skips Codex's `z14144_w045`), 2x m7 snap.

In [ ]:
require_gate(GATE_REVIEWED)
assert not set(PHERC0125_TIER1) & set(PHERC0125_EXCLUDED)
run_streaming(
    build_screen_cmd(
        "PHerc0125",
        [f"{PSCAMILLO_BASE}/{name}" for name in PHERC0125_TIER1],
        snap_passes=2,
    )
)

## (iii) Rank-2 `PHerc1203`: 6 clean S3 `segments/raw/` sheets
`46.66 cm2`, `0%` overlap with Codex A/B/C and Claude N6, `--snap-passes 2`.

In [ ]:
require_gate(GATE_REVIEWED)
run_streaming(
    build_screen_cmd(
        "PHerc1203",
        [f"{PHERC1203_BASE}/{name}" for name in PHERC1203_CLEAN],
        snap_passes=2,
    )
)

## (iv) `PHerc0191` spiral-fit windings `w020`, `w030`, `w040`, `w050`
Public `rodriguescarson/eligible-scroll-spiral-fits` run `slice-11600-12400` (pinned HF revision, CC BY-NC 4.0, credit Carson Rodrigues). The author publishes these as **unvalidated geometry**, so treat the on-sheet score as the first thing to check. 9.362 um S3 Zarr v2 CT + `m7`, `--snap-passes 2`. CPU triage of these four windings: `gemini_lane/evidence/pherc0125_0191_onsheet_triage.json`.

In [ ]:
require_gate(GATE_REVIEWED)
run_streaming(
    build_screen_cmd(
        "PHerc0191",
        [f"{PHERC0191_BASE}/{name}" for name in PHERC0191_WINDINGS],
        snap_passes=2,
    )
)

## (v) Progress Prize FP16 upstream parity burst (1-click Studio + Hecate)
1. Streams the two public uint8 `[64, 256, 256]` Hecate parity crops (`< 15 MB` total) via `gemini_lane/progress_prize_fp16/prepare_hecate_parity_crops.py --out-dir /content/scrollprize_private/hecate_parity_crops` (`hecate_2p4um_crop.npy` and `hecate_9p6um_crop.npy`).
2. Runs `gemini_lane/progress_prize_fp16/verify_upstream_gpu_burst.py --stages studio,hecate` feeding both generated crops (public controls only; no unread data). The sanitized JSON receipt is written to a fresh timestamped file under `/content/scrollprize_private`.

In [ ]:
run_burst()

## (vi) `PHerc0125` Tier-1B high-coherence untouched windings
CPU-triaged `23.06 cm2` (`f_parallel` `0.894-0.985`, `on_sheet_after_snap` `0.958-0.993`): `z9936_w035`, `z9344_w015`, `z11136_w035`, `z15936_w035`, `z8144_w030`, `z14144_w050` (excludes Tier-1 and Codex's `z14144_w045`), `--snap-passes 2`. CPU triage: `gemini_lane/evidence/pherc0125_t1b_1203sep23_onsheet_triage.json`.

In [ ]:
require_gate(GATE_REVIEWED)
assert not set(PHERC0125_TIER1B) & (set(PHERC0125_TIER1) | set(PHERC0125_EXCLUDED))
run_streaming(
    build_screen_cmd(
        "PHerc0125",
        [f"{PSCAMILLO_BASE}/{name}" for name in PHERC0125_TIER1B],
        snap_passes=2,
    )
)

## (vii) `PHerc1203` 7th clean Sep-23 S3 sheet
`auto_grown_20250923163217042` (`3.79 cm2`, `f_parallel` `0.9286`, `on_sheet_after_snap` `0.9533`, `0%` overlap with Codex A/B/C, Claude N6, and top-9 S3 sheets), `--snap-passes 2`. CPU triage: `gemini_lane/evidence/pherc0125_t1b_1203sep23_onsheet_triage.json`.

In [ ]:
require_gate(GATE_REVIEWED)
assert not set(PHERC1203_SEP23_CLEAN) & set(PHERC1203_CLEAN)
run_streaming(
    build_screen_cmd(
        "PHerc1203",
        [f"{PHERC1203_BASE}/{name}" for name in PHERC1203_SEP23_CLEAN],
        snap_passes=2,
    )
)

## (viii) `PHerc0826` spiral-fit windings `w030`, `w050`, `w040`
Public `rodriguescarson/eligible-scroll-spiral-fits` run `slice-8928-9728` (pinned HF revision, CC BY-NC 4.0, credit Carson Rodrigues; `7.76 cm2`, `f_parallel` `0.820-0.926`, `on_sheet_after_snap` `0.858-0.940`; skips low-coherence `w020` and `w027`/`w051`), 9.362 um S3 Zarr v2 CT + `m7`, `--snap-passes 2`. CPU triage: `gemini_lane/evidence/pherc0826_1203t2_onsheet_triage.json`.

In [ ]:
require_gate(GATE_REVIEWED)
assert not set(PHERC0826_WINDINGS) & set(PHERC0826_EXCLUDED)
run_streaming(
    build_screen_cmd(
        "PHerc0826",
        [f"{PHERC0826_BASE}/{name}" for name in PHERC0826_WINDINGS],
        snap_passes=2,
    )
)

## (ix) `PHerc1203` Tier-2 disjoint strips
CPU-triaged `6.82 cm2` (`f_parallel` `0.786-0.966`, `on_sheet_after_snap` `0.925-0.977`) disjoint from Claude N6 and Codex A/B/C: `auto_grown_20251005230830030` (`0:165,0:68`) and `auto_grown_20251005221856743` (`0:253,158:226` and `0:253,0:62`), run one `--uv-crop` per invocation with `--snap-passes 2`. CPU triage: `gemini_lane/evidence/pherc0826_1203t2_onsheet_triage.json`.

In [ ]:
require_gate(GATE_REVIEWED)
for name, uv_crop in PHERC1203_TIER2_STRIPS:
    run_streaming(
        build_screen_cmd(
            "PHerc1203",
            [f"{PHERC1203_BASE}/{name}"],
            snap_passes=2,
            uv_crop=uv_crop,
        )
    )

## 3. Save compact private outputs (outside git)
Zip contains only `montage.png`, `hotspots.png`, `summary.json`. Download it and/or back it up to Drive.

In [ ]:
assert_repo_clean()
zip_path, members = package_private_outputs()
print(zip_path, len(members), "files")
print("\n".join(members))

In [ ]:
BACKUP_TO_DRIVE = False   # True: copy the zip to MyDrive/Scrollprize_private
DOWNLOAD_ZIP = False      # True: browser download of the zip
if BACKUP_TO_DRIVE:
    print(backup_to_drive())
if DOWNLOAD_ZIP and colab_files is not None:
    colab_files.download(str(ZIP_PATH))